# 3. SMOTE + Random Forest

학습 데이터에만 SMOTE 오버샘플링을 적용한 뒤 Random Forest를 학습합니다. 테스트 데이터는 원래 분포를 유지합니다.

각 노트북은 학습/테스트 분할부터 독립 실행할 수 있습니다. 모델 전처리는 학습 데이터에만 맞춰 테스트 데이터 누수를 방지합니다.

## 공통 데이터 준비

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

data_dir = Path.cwd() / "data"
if not data_dir.is_dir():
    data_dir = Path.cwd().parent / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")

X_raw = pd.read_csv(data_dir / "secom.data", sep=r"\s+", header=None)
labels = pd.read_csv(data_dir / "secom_labels.data", sep=r"\s+", header=None)
y = labels.iloc[:, 0].astype(int)

if len(X_raw) != len(y):
    raise ValueError(f"센서 데이터와 라벨 행 수가 다릅니다: {len(X_raw)} != {len(y)}")

# 분할 후 학습 데이터에서만 결측률 기준과 중앙값을 학습해 데이터 누수를 막습니다.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
keep_columns = X_train_raw.columns[X_train_raw.isna().mean() < 0.8]
X_train_raw = X_train_raw.loc[:, keep_columns]
X_test_raw = X_test_raw.loc[:, keep_columns]

imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=keep_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=keep_columns,
    index=X_test_raw.index,
)

keep_columns = X_train.columns[X_train.nunique(dropna=False) > 1]
X_train = X_train.loc[:, keep_columns]
X_test = X_test.loc[:, keep_columns]

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print("학습 라벨 분포:")
print(y_train.value_counts().sort_index())


학습 데이터: (1253, 466), 테스트 데이터: (314, 466)
학습 라벨 분포:
0
-1    1170
 1      83
Name: count, dtype: int64


## 학습 데이터 오버샘플링

In [2]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("SMOTE 적용 전:")
print(y_train.value_counts().sort_index())
print("\nSMOTE 적용 후:")
print(y_train_smote.value_counts().sort_index())

SMOTE 적용 전:
0
-1    1170
 1      83
Name: count, dtype: int64

SMOTE 적용 후:
0
-1    1170
 1    1170
Name: count, dtype: int64


## SMOTE 학습 데이터로 Random Forest 평가

In [3]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

LABELS = [-1, 1]
TARGET_NAMES = ["PASS (-1)", "FAIL (1)"]


def report_results(y_true, y_pred):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=TARGET_NAMES,
        zero_division=0,
    ))
    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=LABELS))


rf_smote = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
rf_smote.fit(X_train_smote, y_train_smote)
pred_smote = rf_smote.predict(X_test)

report_results(y_test, pred_smote)

Accuracy: 0.9299
              precision    recall  f1-score   support

   PASS (-1)       0.93      1.00      0.96       293
    FAIL (1)       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

Confusion matrix (actual rows, predicted columns):
[[292   1]
 [ 21   0]]


In [4]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = data_dir.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def normalize_labels(values):
    values = pd.Series(values).reset_index(drop=True)
    unique_values = set(values.dropna().unique())

    if unique_values.issubset({-1, 1}):
        values = values.map({-1: 0, 1: 1})

    return values.astype(int)


MODEL_NAME = "03SMOTE_RF"
y_true_saved = normalize_labels(y_test)
y_pred_saved = normalize_labels(pred_smote)

result_df = pd.DataFrame({
    "y_true": y_true_saved,
    "y_pred": y_pred_saved,
})

result_path = RESULTS_DIR / f"{MODEL_NAME}.csv"
result_df.to_csv(result_path, index=False)

print(f"저장 완료: {result_path.name}")
print(result_df.head())

저장 완료: 03SMOTE_RF.csv
   y_true  y_pred
0       0       0
1       0       0
2       0       0
3       0       0
4       0       0
